# Module 4: Failure Lab - PostgreSQL

## ⚠️ CRITICAL SAFETY WARNING

**THIS NOTEBOOK WILL MODIFY YOUR ENVIRONMENT:**
- **Modifies Kubernetes secrets** (breaks PostgreSQL password)
- **Causes service disruptions** (API failures, login failures)
- **Requires remediation** to restore functionality

**REQUIREMENTS:**
- ✅ **TEST/NON-PRODUCTION environment ONLY**
- ✅ **MODULE4_SAFE_ENVIRONMENT=true** must be set in .env
- ✅ **Baseline diagnostics collected** (run `01_diagnostics_baseline.ipynb` first)
- ✅ **Backup/restore plan** available

**DO NOT RUN THIS LAB AGAINST PRODUCTION SYSTEMS.**

## Overview

**This lab teaches you how to debug PostgreSQL connectivity failures in LangSmith.**

PostgreSQL is LangSmith's primary metadata store. It holds:
- User accounts and workspaces
- Project definitions
- API keys and permissions
- Trace metadata (not the traces themselves, which go to ClickHouse)

**When PostgreSQL fails, you'll see:**
- API endpoints return 5xx errors
- Login/authentication may fail
- UI may load but actions fail
- Connection exhaustion patterns in logs

**Learning Objectives:**
1. Understand how PostgreSQL failures manifest
2. Practice collecting diagnostics for database issues
3. Learn to identify connection vs. credential vs. network issues
4. Practice safe remediation

**Estimated time:** 30-45 minutes

**⚠️ Important:** 
- Run `01_diagnostics_baseline.ipynb` BEFORE starting this lab!
- Complete safety check in `../shared/00_setup_or_resume_environment.ipynb` first!


In [ ]:
# Bootstrap environment
import sys
from pathlib import Path

# Add notebooks directory to path
possible_paths = [
    Path.cwd().parent,
    Path.cwd(),
    Path.cwd() / "notebooks",
]

notebooks_path = None
for path in possible_paths:
    if path and (path / "shared" / "_bootstrap.py").exists():
        notebooks_path = path
        break

if not notebooks_path:
    notebooks_path = Path.cwd() / "notebooks"
    if not (notebooks_path / "shared" / "_bootstrap.py").exists():
        raise RuntimeError(f"Could not find notebooks/shared directory. Current dir: {Path.cwd()}")

if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

from shared._bootstrap import bootstrap
from shared._validation import ok, warn

# Run bootstrap
bootstrap_info = bootstrap()
artifacts_dir = Path(bootstrap_info['artifacts_dir'])
print(f"\nArtifacts directory: {artifacts_dir}")


## ⚠️ CRITICAL: Environment Safety Verification

**Before proceeding, verify you're in a TEST/NON-PRODUCTION environment and understand what will be modified.**


In [ ]:
# CRITICAL SAFETY CHECK: Verify environment is safe for failure injection
from shared._cloud_helpers import get_cloud_provider, get_region, get_identity
from shared._validation import ok, warn, fail
import os

provider = get_cloud_provider()
region = get_region()
identity = get_identity()

print("=" * 70)
print("⚠️  CRITICAL SAFETY CHECK - POSTGRESQL FAILURE LAB")
print("=" * 70)

# Show environment details prominently
provider_display = provider.upper()
print(f"\n### Current Environment Configuration")
print(f"Cloud Provider: {provider_display}")
print(f"Region: {region}")

if provider == "aws":
    account_id = identity.get('Account', 'N/A')
    user_arn = identity.get('Arn', 'N/A')
    print(f"Account ID: {account_id}")
    print(f"User ARN: {user_arn}")
elif provider == "azure":
    subscription_id = identity.get("SubscriptionId") or identity.get("Account", "N/A")
    subscription_name = identity.get("SubscriptionName", "N/A")
    print(f"Subscription ID: {subscription_id}")
    print(f"Subscription Name: {subscription_name}")

# Show all relevant environment variables
print(f"\n### Environment Variables (VERIFY THESE ARE CORRECT)")
print(f"NAMESPACE: {os.environ.get('NAMESPACE', 'NOT SET')}")
print(f"CLUSTER_NAME: {os.environ.get('CLUSTER_NAME', 'NOT SET')}")
print(f"HELM_RELEASE: {os.environ.get('HELM_RELEASE', 'langsmith')}")
print(f"LANGSMITH_DOMAIN: {os.environ.get('LANGSMITH_DOMAIN', 'NOT SET')}")

print("\n" + "=" * 70)
print("⚠️  WHAT THIS LAB WILL DO:")
print("=" * 70)
print("\nThis failure lab will:")
print("  1. Find the PostgreSQL secret in your namespace")
print("  2. BACKUP the original secret (saved to artifacts)")
print("  3. MODIFY the secret to set an INVALID password")
print("  4. Apply the modified secret (breaks database connectivity)")
print("  5. Cause API failures and login failures")
print("  6. Require remediation to restore (restore original secret)")
print("\n" + "=" * 70)

# Check for Module 4 safety flag
module4_safe = os.environ.get("MODULE4_SAFE_ENVIRONMENT", "").lower()
if module4_safe not in ["true", "yes", "1"]:
    fail("MODULE4_SAFE_ENVIRONMENT flag is NOT set")
    print("\n❌ SAFETY CHECK FAILED - Cannot proceed")
    print("\nTo run this failure lab, you MUST:")
    print("  1. Verify this is a TEST/NON-PRODUCTION environment")
    print("  2. Set MODULE4_SAFE_ENVIRONMENT=true in your .env file")
    print("  3. Complete safety check in ../shared/00_setup_or_resume_environment.ipynb")
    print("  4. Re-run this cell to confirm")
    print("\nThis flag is REQUIRED to prevent accidental execution in production.")
    raise RuntimeError("MODULE4_SAFE_ENVIRONMENT not set. Required for failure labs.")

ok("MODULE4_SAFE_ENVIRONMENT flag is set")
print("\n✅ Safety check passed - environment marked as safe for failure injection")
print("\n⚠️  REMINDER: This lab will break PostgreSQL connectivity.")
print("   Ensure you understand the remediation steps before proceeding.")
print("   Original secret will be backed up automatically.")

print("\n" + "=" * 70)
print("✅ Environment verified - ready for PostgreSQL failure lab")
print("=" * 70)


## 1. Configuration & Prerequisites

Load configuration and verify prerequisites.


In [ ]:
import os
from shared._validation import require_env

required_vars = ["NAMESPACE", "CLUSTER_NAME"]
config = require_env(*required_vars)
config["HELM_RELEASE"] = os.environ.get("HELM_RELEASE", "langsmith")

namespace = config["NAMESPACE"]

print(f"Namespace: {namespace}")
print(f"Helm Release: {config['HELM_RELEASE']}")

ok("Configuration loaded")


## 2. What This Service Does for LangSmith

PostgreSQL is LangSmith's **primary metadata store**. It holds:

- **User accounts and authentication data**
- **Workspaces and projects** (organizational structure)
- **API keys and permissions** (access control)
- **Trace metadata** (not the trace data itself, which goes to ClickHouse)
- **Evaluation results and feedback**

**Why it matters:**
- Without PostgreSQL, users can't log in
- API calls fail (no authentication, no project lookups)
- UI loads but can't perform actions
- All LangSmith functionality depends on it

**How LangSmith connects:**
- Connection string stored in Kubernetes Secrets
- Connection pool managed by application
- Connection limits are critical (PostgreSQL has max connections)


## 3. Expected Symptoms When PostgreSQL Fails

**What you'll see:**

1. **API 5xx errors:**
   - `/api/v1/...` endpoints return 500 or 503
   - Error messages mention "database" or "connection"

2. **Login failures:**
   - Users can't authenticate
   - OIDC/SAML may work (redirects) but session creation fails

3. **UI loads but actions fail:**
   - Pages render (static content)
   - API calls fail (can't load projects, traces, etc.)

4. **Log patterns:**
   - Connection timeout errors
   - "connection refused" or "connection reset"
   - "too many connections" (if connection pool exhausted)
   - "authentication failed" (if credentials wrong)

**Timeline:**
- Symptoms appear within seconds of failure
- API calls start failing immediately
- Existing connections may work briefly, then fail


## 4. Failure Injection Options

**Choose ONE level to practice with. Level 1 is subtle, Level 2 is more obvious.**

### Level 1: Subtle Failure (Recommended for first run)

**Option A: Wrong Database Password**
- Modify the PostgreSQL password in the Kubernetes Secret
- Symptoms: Authentication failures, connection refused

**Option B: Wrong Database Host**
- Point connection string to non-existent host
- Symptoms: Connection timeout, DNS resolution failures

**Option C: Network Isolation (if NetworkPolicy supported)**
- Apply NetworkPolicy blocking egress to PostgreSQL
- Symptoms: Connection timeout, no route to host

### Level 2: Obvious Failure

**Option D: Remove Secret Entirely**
- Delete the PostgreSQL connection secret
- Symptoms: Pods crash on startup, immediate failures

**⚠️ Safety:** All injections are reversible. We'll save the original secret before modifying it.


## 5. Do the Drill - Step 1: Confirm Baseline

**Before injecting any failure, verify your baseline is healthy.**

💡 **If you haven't run `01_diagnostics_baseline.ipynb` yet, do that first!**


In [ ]:
from shared._shell import run
import json

print("### Quick Baseline Check\n")

# Check pod status
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    pods = json.loads(result.stdout)
    healthy = sum(1 for p in pods.get("items", [])
                  if p.get("status", {}).get("phase") == "Running")
    total = len(pods.get("items", []))
    print(f"Pods: {healthy}/{total} running")
    
    if healthy == total and total > 0:
        ok("Baseline looks healthy")
    else:
        warn("Some pods are not running - check baseline first")
else:
    warn("Could not check pod status")

# Check for PostgreSQL secret
result = run(
    ["kubectl", "get", "secrets", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

postgres_secrets = []
if result.returncode == 0:
    secrets = json.loads(result.stdout)
    for secret in secrets.get("items", []):
        name = secret.get("metadata", {}).get("name", "")
        if "postgres" in name.lower() or "database" in name.lower() or "db" in name.lower():
            postgres_secrets.append(name)

if postgres_secrets:
    ok(f"Found {len(postgres_secrets)} PostgreSQL-related secret(s)")
    for secret_name in postgres_secrets:
        print(f"   - {secret_name}")
else:
    warn("No PostgreSQL secrets found")
    print("   💡 PostgreSQL connection may be configured differently")


## 6. Do the Drill - Step 2: Apply Failure Injection

**⚠️ WARNING: This will modify your LangSmith deployment. Make sure you're in a test environment!**

Choose your failure injection method below. We'll use **Option A (Wrong Password)** as the default example.


In [ ]:
# FAILURE INJECTION: Wrong Database Password
# This cell modifies the PostgreSQL password secret to an invalid value

import base64
import yaml
from datetime import datetime

# Find PostgreSQL secret (look for common names)
result = run(
    ["kubectl", "get", "secrets", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

postgres_secret_name = None
if result.returncode == 0:
    secrets = json.loads(result.stdout)
    for secret in secrets.get("items", []):
        name = secret.get("metadata", {}).get("name", "")
        # Common patterns: postgres, database, db, langsmith-db
        if any(keyword in name.lower() for keyword in ["postgres", "database", "db"]):
            # Check if it has password-related keys
            data = secret.get("data", {})
            if any(key in data for key in ["password", "POSTGRES_PASSWORD", "DB_PASSWORD"]):
                postgres_secret_name = name
                break

if not postgres_secret_name:
    raise RuntimeError("❌ Could not find PostgreSQL secret. Check your deployment configuration.")

print(f"Found PostgreSQL secret: {postgres_secret_name}")

# Get current secret
result = run(
    ["kubectl", "get", "secret", postgres_secret_name, "-n", namespace, "-o", "yaml"],
    check=True,
    stream=False
)

# Save original secret for restoration
backup_file = artifacts_dir / "module-4" / f"postgres-secret-backup-{datetime.now().strftime('%Y%m%d-%H%M%S')}.yaml"
backup_file.parent.mkdir(parents=True, exist_ok=True)
with open(backup_file, "w") as f:
    f.write(result.stdout)

ok(f"Backed up original secret to: {backup_file.name}")

# Parse YAML and modify password
secret_data = yaml.safe_load(result.stdout)
if "data" not in secret_data:
    raise RuntimeError("Secret has no data section")

# Find password key (could be password, POSTGRES_PASSWORD, DB_PASSWORD, etc.)
password_key = None
for key in ["password", "POSTGRES_PASSWORD", "DB_PASSWORD", "postgres-password"]:
    if key in secret_data["data"]:
        password_key = key
        break

if not password_key:
    raise RuntimeError("Could not find password key in secret")

# Set invalid password
invalid_password = "INVALID_PASSWORD_12345"
invalid_password_b64 = base64.b64encode(invalid_password.encode()).decode()

# Modify secret
secret_data["data"][password_key] = invalid_password_b64

# Save modified secret to temp file
temp_secret_file = artifacts_dir / "module-4" / "postgres-secret-modified.yaml"
with open(temp_secret_file, "w") as f:
    yaml.dump(secret_data, f)

print("=" * 70)
print("⚠️  READY TO APPLY FAILURE INJECTION")
print("=" * 70)
print(f"\nThis will modify secret: {postgres_secret_name}")
print(f"Modified secret saved to: {temp_secret_file.name}")
print(f"Backup saved to: {backup_file.name}")
print("\n" + "=" * 70)
print("⚠️  FINAL WARNING BEFORE FAILURE INJECTION")
print("=" * 70)
print("\nThis will:")
print("  ❌ Break PostgreSQL connectivity")
print("  ❌ Cause API 5xx errors")
print("  ❌ Break login/authentication")
print("  ❌ Disrupt LangSmith functionality")
print("\nTo apply the failure:")
print("  1. Verify MODULE4_SAFE_ENVIRONMENT=true is set")
print("  2. Verify you're in a TEST environment")
print("  3. Uncomment the code in the next cell")
print("  4. Run the next cell to apply")
print("\nTo restore after the lab:")
print(f"  - Use the backup file: {backup_file.name}")
print("  - See the 'Remediation' section below")
print("\n" + "=" * 70)


In [ ]:
# UNCOMMENT TO APPLY FAILURE INJECTION
# 
# result = run(
#     ["kubectl", "apply", "-f", str(temp_secret_file)],
#     check=True,
#     stream=True
# )
# 
# ok("Failure injection applied - PostgreSQL password is now invalid")
# print("\n💡 Pods will need to restart to pick up the new secret.")
# print("   This may take 1-2 minutes. Watch for pod restarts:")
# print(f"   kubectl get pods -n {namespace} -w")
# 
# # Wait a moment for changes to propagate
# import time
# print("\nWaiting 30 seconds for changes to propagate...")
# time.sleep(30)


## 8. Do the Drill - Step 3: Observe Symptoms

**Now that the failure is injected, observe how it manifests.**

Check:
1. Pod logs for connection errors
2. API endpoint responses
3. UI behavior
4. Events for pod restarts


In [ ]:
from datetime import datetime

# Create incident directory for diagnostics
incident_dir = artifacts_dir / "module-4" / f"postgres-failure-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
incident_dir.mkdir(parents=True, exist_ok=True)

print(f"### Collecting Failure Diagnostics\n")
print(f"Saving to: {incident_dir}\n")

# 1. Check pod status
print("1. Checking pod status...")
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "wide"],
    check=False,
    stream=False
)

if result.returncode == 0:
    with open(incident_dir / "pods-status.txt", "w") as f:
        f.write(result.stdout)
    print(result.stdout)
    
    # Check for restarts
    lines = result.stdout.split("\n")
    restarts = [l for l in lines if "RESTARTS" in l or (l and not l.startswith("NAME"))]
    if restarts:
        print("\n   Pod restart counts:")
        for line in restarts[1:]:  # Skip header
            if line.strip():
                parts = line.split()
                if len(parts) > 3:
                    print(f"   {parts[0]}: {parts[3]} restarts")

# 2. Check recent events
print("\n2. Checking recent events...")
result = run(
    ["kubectl", "get", "events", "-n", namespace, "--sort-by='.lastTimestamp'", "--field-selector=type!=Normal"],
    check=False,
    stream=False
)

if result.returncode == 0:
    with open(incident_dir / "events.txt", "w") as f:
        f.write(result.stdout)
    if result.stdout.strip():
        print("   Recent warning/error events:")
        for line in result.stdout.split("\n")[-5:]:
            if line.strip():
                print(f"   {line}")

# 3. Check API pod logs for database errors
print("\n3. Checking API pod logs for database errors...")
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-l", "app=langsmith-api", "-o", "jsonpath='{.items[0].metadata.name}'"],
    check=False,
    stream=False
)

api_pod = result.stdout.strip().strip("'\"")
if api_pod:
    result = run(
        ["kubectl", "logs", "-n", namespace, api_pod, "--tail=50"],
        check=False,
        stream=False
    )
    
    if result.returncode == 0:
        logs_file = incident_dir / f"api-pod-{api_pod}-logs.txt"
        with open(logs_file, "w") as f:
            f.write(result.stdout)
        
        # Look for database-related errors
        error_keywords = ["database", "postgres", "connection", "timeout", "refused", "authentication"]
        error_lines = [l for l in result.stdout.split("\n") 
                      if any(kw in l.lower() for kw in error_keywords)]
        
        if error_lines:
            print("   Found database-related errors:")
            for line in error_lines[-5:]:
                print(f"   {line}")
        else:
            print("   No obvious database errors in recent logs")
else:
    warn("Could not find API pod")

ok(f"Diagnostics saved to: {incident_dir}")


## 9. Do the Drill - Step 4: Run Canonical Diagnostics Script

**This is critical - Support will ask for this bundle.**


In [ ]:
import urllib.request

print("### Running Canonical Diagnostics Script\n")

script_url = "https://raw.githubusercontent.com/langchain-ai/helm/main/charts/langsmith/scripts/get_k8s_debugging_info.sh"
script_path = incident_dir / "get_k8s_debugging_info.sh"

try:
    urllib.request.urlretrieve(script_url, script_path)
    script_path.chmod(0o755)
    
    print(f"Running diagnostics script for namespace: {namespace}")
    result = run(
        [str(script_path), namespace],
        check=False,
        stream=True
    )
    
    if result.returncode == 0:
        ok("Diagnostics script completed")
        
        # Find and move tarball
        for file in incident_dir.parent.iterdir():
            if file.name.startswith("langsmith-debug-") and file.suffix == ".tar.gz":
                target_path = incident_dir / file.name
                file.rename(target_path)
                ok(f"Diagnostics bundle: {target_path.name}")
                break
    else:
        warn("Diagnostics script had errors (check output above)")
        
except Exception as e:
    warn(f"Could not run diagnostics script: {e}")
    print("   💡 You can run it manually:")
    print(f"      curl -O {script_url}")
    print(f"      chmod +x get_k8s_debugging_info.sh")
    print(f"      ./get_k8s_debugging_info.sh {namespace}")


## 10. Do the Drill - Step 5: Guided Triage

**Where to look first for PostgreSQL issues:**


In [ ]:
print("### Guided Triage Steps\n")

print("1. Check pod logs for connection errors:")
print(f"   kubectl logs -n {namespace} <pod-name> | grep -i 'database\\|postgres\\|connection'")
print()

print("2. Verify secret exists and has correct keys:")
print(f"   kubectl get secret {postgres_secret_name} -n {namespace} -o yaml")
print("   (Don't print the actual values - they're base64 encoded)")
print()

print("3. Check for pod restarts (indicates startup failures):")
print(f"   kubectl get pods -n {namespace}")
print()

print("4. Test database connectivity from a pod (if possible):")
print("   kubectl run -it --rm debug --image=postgres:15 --restart=Never -- \\")
print("     psql -h <db-host> -U <user> -d <database>")
print()

print("5. Check events for authentication/connection errors:")
print(f"   kubectl get events -n {namespace} --sort-by='.lastTimestamp' | grep -i 'error\\|fail'")
print()

# Check what we can automatically
print("\n### Automatic Checks\n")

# Check secret still exists
result = run(
    ["kubectl", "get", "secret", postgres_secret_name, "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    ok(f"Secret '{postgres_secret_name}' still exists")
    secret_data = json.loads(result.stdout)
    keys = list(secret_data.get("data", {}).keys())
    print(f"   Secret keys: {', '.join(keys)}")
else:
    warn(f"Secret '{postgres_secret_name}' not found!")

# Check for pods with database connection env vars
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    pods = json.loads(result.stdout)
    db_related_pods = []
    for pod in pods.get("items", []):
        name = pod.get("metadata", {}).get("name", "")
        containers = pod.get("spec", {}).get("containers", [])
        for container in containers:
            env = container.get("env", [])
            db_env = [e for e in env if any(kw in e.get("name", "").upper() 
                                           for kw in ["DB", "POSTGRES", "DATABASE"])]
            if db_env:
                db_related_pods.append(name)
                break
    
    if db_related_pods:
        print(f"\n   Pods with database environment variables:")
        for pod_name in set(db_related_pods):
            print(f"   - {pod_name}")


## 11. Do the Drill - Step 6: Remediation

**Restore the original secret to fix the issue.**


In [ ]:
# REMEDIATION: Restore original secret
# UNCOMMENT TO RESTORE

# if backup_file.exists():
#     print(f"Restoring original secret from: {backup_file.name}")
#     result = run(
#         ["kubectl", "apply", "-f", str(backup_file)],
#         check=True,
#         stream=True
#     )
#     
#     ok("Original secret restored")
#     print("\n💡 Pods will restart to pick up the correct secret.")
#     print("   This may take 1-2 minutes. Monitor pod status:")
#     print(f"   kubectl get pods -n {namespace} -w")
#     
#     import time
#     print("\nWaiting 60 seconds for pods to restart...")
#     time.sleep(60)
# else:
#     warn(f"Backup file not found: {backup_file}")
#     print("   💡 You may need to manually restore the secret")

print("⚠️  To restore, uncomment the code above and run this cell.")
print(f"   Backup file: {backup_file.name}")


## 12. Do the Drill - Step 7: Confirm Recovery

**Verify that everything is working again.**


In [ ]:
print("### Verifying Recovery\n")

# Check pod status
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    pods = json.loads(result.stdout)
    running = sum(1 for p in pods.get("items", [])
                  if p.get("status", {}).get("phase") == "Running")
    total = len(pods.get("items", []))
    
    if running == total and total > 0:
        ok(f"All {total} pod(s) are running")
    else:
        warn(f"Only {running}/{total} pod(s) running")
        print("   💡 Wait a bit longer for pods to fully recover")

# Check for recent errors in logs
print("\nChecking for recent errors...")
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-l", "app=langsmith-api", "-o", "jsonpath='{.items[0].metadata.name}'"],
    check=False,
    stream=False
)

api_pod = result.stdout.strip().strip("'\"")
if api_pod:
    result = run(
        ["kubectl", "logs", "-n", namespace, api_pod, "--tail=20"],
        check=False,
        stream=False
    )
    
    if result.returncode == 0:
        error_keywords = ["error", "fail", "database", "postgres", "connection"]
        recent_errors = [l for l in result.stdout.split("\n") 
                        if any(kw in l.lower() for kw in error_keywords)]
        
        if recent_errors:
            warn("Still seeing some errors in logs:")
            for line in recent_errors[-3:]:
                print(f"   {line}")
        else:
            ok("No recent errors in API logs")

ok("Recovery verification complete")


## 13. What Support Will Ask For

**When escalating a PostgreSQL issue, Support will need:**

1. **Diagnostics bundle** (canonical script output) ✅ Collected above
2. **PostgreSQL connection details:**
   - Host/endpoint (redacted)
   - Database name
   - Username (redacted)
   - Whether using SSL/TLS
3. **Error messages from logs:**
   - Full error text (not just "connection failed")
   - Timestamps of first occurrence
4. **Recent changes:**
   - Secret rotations
   - Database migrations
   - Network policy changes
5. **Connection pool status:**
   - Current connections vs. max connections
   - Connection pool exhaustion patterns
6. **Database health (if accessible):**
   - PostgreSQL version
   - Active connections
   - Lock contention

**Evidence collected in this lab:**
- ✅ Diagnostics bundle
- ✅ Pod logs with database errors
- ✅ Events showing failures
- ✅ Secret configuration (structure, not values)

**Additional evidence to gather (if escalating):**
- Database endpoint connectivity test
- Connection pool metrics (if available)
- PostgreSQL logs (if accessible via cloud provider)


## 14. Lessons Learned

**Key takeaways from this lab:**

1. **PostgreSQL failures manifest quickly** - API calls fail within seconds
2. **Logs are your friend** - Connection errors appear in pod logs immediately
3. **Secrets matter** - Wrong credentials cause authentication failures
4. **Baseline is critical** - You need "before" to compare to "after"
5. **Diagnostics bundle is essential** - Support needs it for root cause analysis

**Common mistakes to avoid:**
- ❌ Changing multiple things at once (hard to identify root cause)
- ❌ Not collecting diagnostics before remediation
- ❌ Ignoring connection pool limits
- ❌ Not testing database connectivity independently

**Next steps:**
- Practice with other failure injection methods (Level 2)
- Try the Redis, ClickHouse, or Blob Storage failure labs
- Review the [First 10 Minutes Checklist](../shared/incident_first_10_minutes.md)
